In [121]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd
import sys
import torch
from pathlib import Path

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_root(current_path, marker="setup.py"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    return current_path

PROJECT_ROOT = find_root(Path.cwd())
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
ASSETS_DIR = PROJECT_ROOT / "experiments" / "shared" / "assets"

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rl_methods.sbeed import (
    SBEEDSolver,
    DiscreteMDPSpec,
    TabularStateFeatures,
    TabularStateActionFeatures,
    SBEEDEvaluator,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## MDP

In [46]:
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)

N = len(states)
A = len(actions)
gamma = 0.9

x_0 = 0
goal_grid = 8

In [47]:
def to_rc(s):
    return divmod(int(s), 3)

def to_s(r, c):
    return r * 3 + c

def next_state(s, a):
    s = int(s)
    a = int(a)

    if s == goal_grid:
        return goal_grid

    r, c = to_rc(s)

    if a == 0:      # Up
        r = max(0, r - 1)
    elif a == 1:    # Down
        r = min(2, r + 1)
    elif a == 2:    # Left
        c = max(0, c - 1)
    elif a == 3:    # Right
        c = min(2, c + 1)

    return to_s(r, c)


### Features

In [48]:
value_features = TabularStateFeatures(n_states=N)
rho_features = TabularStateActionFeatures(n_states=N, n_actions=A)

mdp_spec = DiscreteMDPSpec(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    value_features=value_features,
    rho_features=rho_features,
    x0=x_0,
)

def reward_fn(s, a, sp):
    # Same idea as your old omega: reward 1 at the goal.
    return 1.0 if int(sp) == goal_grid else -0.1

## Solving it

In [104]:
solver_sbeed = SBEEDSolver(
    spec=mdp_spec,
    lambda_entropy=0.01,
    eta=0.01,
    ridge=1e-6,
    lr_value=1e-2,
    lr_policy=1,
    tau=100000.0,
    buffer_mode="fifo",
    max_buffer_size=3000,
    batch_size=64,
    seed=SEED,
    device=DEVICE,
)

In [105]:
pi_sbeed = solver_sbeed.run(
    transition_fn=next_state,
    reward_fn=reward_fn,
    episodes=2000,
    collect_per_episode=20,
    updates_per_episode=10,
    initial_collect_steps=50,
    start_state=x_0,
    behavior="policy",
    epsilon=0.1,
    terminal_states={goal_grid},
    tqdm_print=False,
    verbose=True,
    log_every=10,
)

episode=10/2000 buffer=250 objective=0.026091 primal_mse=0.026137 dual_mse=0.004562 theta_grad=8.755e-02 policy_grad=5.725e-04
episode=20/2000 buffer=450 objective=0.023861 primal_mse=0.023933 dual_mse=0.007170 theta_grad=4.428e-02 policy_grad=3.707e-04
episode=30/2000 buffer=650 objective=0.025732 primal_mse=0.025949 dual_mse=0.021718 theta_grad=2.269e-02 policy_grad=2.072e-04
episode=40/2000 buffer=850 objective=0.025225 primal_mse=0.025328 dual_mse=0.010390 theta_grad=6.008e-02 policy_grad=5.461e-04
episode=50/2000 buffer=1050 objective=0.025096 primal_mse=0.025101 dual_mse=0.000499 theta_grad=1.077e-01 policy_grad=6.960e-04
episode=60/2000 buffer=1250 objective=0.023767 primal_mse=0.023989 dual_mse=0.022173 theta_grad=2.492e-02 policy_grad=1.992e-04
episode=70/2000 buffer=1450 objective=0.021743 primal_mse=0.021845 dual_mse=0.010280 theta_grad=4.929e-02 policy_grad=4.392e-04
episode=80/2000 buffer=1650 objective=0.019992 primal_mse=0.020097 dual_mse=0.010575 theta_grad=4.383e-02 po

In [106]:
# Policy matrix: shape (N, A)
pi = solver_sbeed.get_policy_matrix()
pi

tensor([[0.1424, 0.3599, 0.1443, 0.3534],
        [0.2022, 0.3458, 0.1309, 0.3211],
        [0.2236, 0.3859, 0.1702, 0.2203],
        [0.1325, 0.3343, 0.1990, 0.3342],
        [0.1500, 0.3622, 0.1477, 0.3402],
        [0.1153, 0.6169, 0.1173, 0.1504],
        [0.1692, 0.2208, 0.2120, 0.3979],
        [0.1134, 0.1429, 0.1138, 0.6298],
        [0.2500, 0.2500, 0.2500, 0.2500]])

# Wall and pit

In [117]:
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)

N = len(states)
A = len(actions)
gamma = 0.9

x_0 = 0

goal_grid = 8
pit_grid = 5
wall_states = {4}
terminal_states = {goal_grid, pit_grid}

def next_state(s, a):
    s = int(s)
    a = int(a)

    if s in terminal_states:
        return s

    row, col = divmod(s, 3)

    if a == 0:      # up
        new_row, new_col = row - 1, col
    elif a == 1:    # down
        new_row, new_col = row + 1, col
    elif a == 2:    # left
        new_row, new_col = row, col - 1
    elif a == 3:    # right
        new_row, new_col = row, col + 1
    else:
        raise ValueError("action must be in {0,1,2,3}")

    if not (0 <= new_row < 3 and 0 <= new_col < 3):
        return s

    sp = new_row * 3 + new_col

    if sp in wall_states:
        return s

    return sp

value_features = TabularStateFeatures(n_states=N)
rho_features = TabularStateActionFeatures(n_states=N, n_actions=A)

mdp_spec = DiscreteMDPSpec(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    value_features=value_features,
    rho_features=rho_features,
    x0=x_0,
)

def reward_fn(s, a, sp):
    if int(sp) == goal_grid:
        return 1.0
    if int(sp) == pit_grid:
        return -1.0
    return -0.1

In [119]:
solver_sbeed = SBEEDSolver(
    spec=mdp_spec,
    lambda_entropy=0.01,
    eta=0.01,
    ridge=1e-6,
    lr_value=1e-2,
    lr_policy=1,
    tau=100000.0,
    buffer_mode="fifo",
    max_buffer_size=6000,
    batch_size=128,
    seed=SEED,
    device=DEVICE,
)

pi_sbeed = solver_sbeed.run(
    transition_fn=next_state,
    reward_fn=reward_fn,
    episodes=2000,
    collect_per_episode=20,
    updates_per_episode=10,
    initial_collect_steps=50,
    start_state=x_0,
    behavior="policy",
    epsilon=0.2,
    terminal_states={goal_grid},
    tqdm_print=False,
    verbose=True,
    log_every=50,
)

episode=50/2000 buffer=1050 objective=0.167652 primal_mse=0.167791 dual_mse=0.013893 theta_grad=2.114e-02 policy_grad=5.131e-04
episode=100/2000 buffer=2050 objective=0.143658 primal_mse=0.143974 dual_mse=0.031602 theta_grad=5.298e-02 policy_grad=8.992e-04
episode=150/2000 buffer=3050 objective=0.137333 primal_mse=0.137417 dual_mse=0.008388 theta_grad=2.599e-02 policy_grad=7.324e-04
episode=200/2000 buffer=4050 objective=0.132083 primal_mse=0.132429 dual_mse=0.034682 theta_grad=2.428e-02 policy_grad=6.748e-04
episode=250/2000 buffer=5050 objective=0.127061 primal_mse=0.127075 dual_mse=0.001449 theta_grad=2.366e-02 policy_grad=6.264e-04
episode=300/2000 buffer=6000 objective=0.129695 primal_mse=0.129702 dual_mse=0.000742 theta_grad=3.487e-02 policy_grad=4.803e-04
episode=350/2000 buffer=6000 objective=0.127665 primal_mse=0.127890 dual_mse=0.022497 theta_grad=2.825e-02 policy_grad=8.064e-04
episode=400/2000 buffer=6000 objective=0.124074 primal_mse=0.124288 dual_mse=0.021445 theta_grad=3

In [123]:
evaluator = SBEEDEvaluator(
    solver_sbeed,
    next_state_fn=next_state,
    reward_fn=reward_fn,
)
evaluator.print_policy()



========== SBEED POLICY ==========

State 0: pi(0|0)=0.096  pi(1|0)=0.758  pi(2|0)=0.094  pi(3|0)=0.053  --> best action: 1
State 1: pi(0|1)=0.250  pi(1|1)=0.247  pi(2|1)=0.393  pi(3|1)=0.109  --> best action: 2
State 2: pi(0|2)=0.260  pi(1|2)=0.042  pi(2|2)=0.440  pi(3|2)=0.258  --> best action: 2
State 3: pi(0|3)=0.042  pi(1|3)=0.783  pi(2|3)=0.087  pi(3|3)=0.088  --> best action: 1
State 4: pi(0|4)=0.250  pi(1|4)=0.250  pi(2|4)=0.250  pi(3|4)=0.250  --> best action: 0
State 5: pi(0|5)=0.080  pi(1|5)=0.509  pi(2|5)=0.124  pi(3|5)=0.287  --> best action: 1
State 6: pi(0|6)=0.043  pi(1|6)=0.083  pi(2|6)=0.083  pi(3|6)=0.791  --> best action: 3
State 7: pi(0|7)=0.072  pi(1|7)=0.071  pi(2|7)=0.041  pi(3|7)=0.815  --> best action: 3
State 8: pi(0|8)=0.250  pi(1|8)=0.250  pi(2|8)=0.250  pi(3|8)=0.250  --> best action: 0




In [125]:
stats = evaluator.compare_to_optimal_values(print_each=False)

l2_to_v_lambda = stats["l2_to_soft"]
print("||V_sbeed - V_lambda*||_2:", l2_to_v_lambda)


||V_sbeed - V_lambda*||_2: 26.46982759241936


# 5 grid

## Deterministic

In [126]:
states = torch.arange(25, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)

N = len(states)
A = len(actions)
gamma = 0.9

grid_size = 5

x_0 = 0

goal_grid = 24
pit_grid = 18

wall_states = {6, 7, 12}
terminal_states = {goal_grid, pit_grid}

def state_to_pos(s):
    s = int(s)
    return divmod(s, grid_size)


def pos_to_state(row, col):
    return row * grid_size + col


def move_deterministic(s, a):
    """
    Deterministic transition used internally.

    This function returns the next state after applying action a.
    It does not return probabilities.
    """
    s = int(s)
    a = int(a)

    if s in terminal_states:
        return s

    row, col = state_to_pos(s)

    if a == 0:      # up
        new_row, new_col = row - 1, col
    elif a == 1:    # down
        new_row, new_col = row + 1, col
    elif a == 2:    # left
        new_row, new_col = row, col - 1
    elif a == 3:    # right
        new_row, new_col = row, col + 1
    else:
        raise ValueError("action must be in {0,1,2,3}")

    if not (0 <= new_row < grid_size and 0 <= new_col < grid_size):
        return s

    sp = pos_to_state(new_row, new_col)

    if sp in wall_states:
        return s

    return sp


def transition_probs(s, a):
    """
    Return transition distribution for state-action pair.

    Returns:
        List of (next_state, probability)
    """
    sp = move_deterministic(s, a)
    return [(sp, 1.0)]


def next_state(s, a):
    """
    Sample or select next state.

    For deterministic transitions this just returns the only possible next state.
    This keeps compatibility with your previous code.
    """
    probs = transition_probs(s, a)
    return probs[0][0]


def reward_fn(s, a, sp):
    sp = int(sp)

    if sp == goal_grid:
        return 1.0

    if sp == pit_grid:
        return -1.0

    return -0.1


value_features = TabularStateFeatures(n_states=N)
rho_features = TabularStateActionFeatures(n_states=N, n_actions=A)

mdp_spec = DiscreteMDPSpec(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    value_features=value_features,
    rho_features=rho_features,
    x0=x_0,
)

In [129]:
solver_sbeed = SBEEDSolver(
    spec=mdp_spec,
    lambda_entropy=0.01,
    eta=0.01,
    ridge=1e-6,
    lr_value=1e-2,
    lr_policy=1,
    tau=1000.0,
    buffer_mode="fifo",
    max_buffer_size=12000,
    batch_size=256,
    seed=SEED,
    device=DEVICE,
)

pi_sbeed = solver_sbeed.run(
    transition_fn=next_state,
    reward_fn=reward_fn,
    episodes=2000,
    collect_per_episode=20,
    updates_per_episode=10,
    initial_collect_steps=50,
    start_state=x_0,
    behavior="policy",
    epsilon=0.2,
    terminal_states={goal_grid},
    tqdm_print=False,
    verbose=True,
    log_every=50,
)

episode=50/2000 buffer=1050 objective=0.040995 primal_mse=0.041018 dual_mse=0.002262 theta_grad=1.446e-02 policy_grad=2.078e-04
episode=100/2000 buffer=2050 objective=0.042974 primal_mse=0.043039 dual_mse=0.006495 theta_grad=1.985e-02 policy_grad=2.512e-04
episode=150/2000 buffer=3050 objective=0.040664 primal_mse=0.040702 dual_mse=0.003882 theta_grad=1.871e-02 policy_grad=1.672e-04
episode=200/2000 buffer=4050 objective=0.039244 primal_mse=0.039265 dual_mse=0.002043 theta_grad=1.722e-02 policy_grad=1.921e-04
episode=250/2000 buffer=5050 objective=0.036955 primal_mse=0.037011 dual_mse=0.005641 theta_grad=2.255e-02 policy_grad=2.373e-04
episode=300/2000 buffer=6050 objective=0.033217 primal_mse=0.033299 dual_mse=0.008244 theta_grad=1.803e-02 policy_grad=2.212e-04
episode=350/2000 buffer=7050 objective=0.031949 primal_mse=0.032008 dual_mse=0.005899 theta_grad=1.534e-02 policy_grad=1.670e-04
episode=400/2000 buffer=8050 objective=0.033676 primal_mse=0.033741 dual_mse=0.006529 theta_grad=1

KeyboardInterrupt: 

## Stochastic

In [ ]:
def transition_probs(s, a):
    """
    Stochastic transition:
    80% intended action
    20% random action uniformly over all actions.
    """
    s = int(s)
    a = int(a)

    probs_by_state = {}

    for candidate_a in range(A):
        if candidate_a == a:
            prob = 0.8 + 0.2 / A
        else:
            prob = 0.2 / A

        sp = move_deterministic(s, candidate_a)
        probs_by_state[sp] = probs_by_state.get(sp, 0.0) + prob

    return list(probs_by_state.items())

def next_state(s, a):
    probs = transition_probs(s, a)

    next_states = [sp for sp, p in probs]
    probabilities = torch.tensor([p for sp, p in probs], dtype=torch.float)

    idx = torch.multinomial(probabilities, num_samples=1).item()

    return next_states[idx]